In [ ]:
# Load Nigeria climate data
nigeria_path = '../data/nigeria.csv' if not os.path.exists('data/nigeria.csv') else 'data/nigeria.csv'
df = pd.read_csv(nigeria_path)
df['Country'] = 'Nigeria'
# Convert YEAR and DOY to datetime
df['date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['date'].dt.month
df.head()

## 2. Data Loading & Date Parsing

We will load the Nigeria climate data, add a Country column, and convert YEAR and DOY to a proper datetime column. We will also extract the Month for seasonal analysis.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats
import os

# Nigeria Climate Data EDA

This notebook performs exploratory data analysis (EDA), cleaning, and visualization for Nigeria's climate dataset as part of the African Climate Trend Analysis challenge.

---

## 1. Import Required Libraries
We begin by importing the necessary Python libraries for data manipulation and visualization.

# Nigeria Climate EDA
Task 2 notebook for Nigeria.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

In [ ]:
country = 'Nigeria'
df = pd.read_csv('../data/nigeria.csv')
df['Country'] = country
df['DATE'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['DATE'].dt.month
df = df.replace(-999, np.nan)
dup_count = int(df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicate rows found and removed:', dup_count)

In [1]:
display(df.describe(include='number'))
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df)) * 100
display(missing.sort_values('missing_pct', ascending=False))

NameError: name 'df' is not defined

In [ ]:
z_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z_input = df[z_cols].copy().fillna(df[z_cols].median(numeric_only=True))
z_abs = np.abs(zscore(z_input, nan_policy='omit'))
outlier_rows = (z_abs > 3).any(axis=1)
print('Outlier rows (|Z| > 3):', int(outlier_rows.sum()))

In [ ]:
row_missing_ratio = df.isna().mean(axis=1)
df = df[row_missing_ratio <= 0.30].copy()
weather_cols = ['T2M','T2M_MAX','T2M_MIN','T2M_RANGE','PRECTOTCORR','RH2M','WS2M','WS2M_MAX','PS','QV2M']
for col in weather_cols:
    if col in df.columns:
        df[col] = df[col].ffill()
df.to_csv('../data/nigeria_clean.csv', index=False)
print('Saved cleaned file to ../data/nigeria_clean.csv')

In [ ]:
monthly_temp = df.resample('M', on='DATE')['T2M'].mean()
plt.figure(figsize=(12, 4))
monthly_temp.plot()
plt.title('Monthly Average T2M - Nigeria')
plt.ylabel('Temperature (°C)')
plt.show()

monthly_prec = df.resample('M', on='DATE')['PRECTOTCORR'].sum()
plt.figure(figsize=(12, 4))
monthly_prec.plot(kind='bar')
plt.title('Monthly Total PRECTOTCORR - Nigeria')
plt.ylabel('Precipitation (mm/month)')
plt.tight_layout()
plt.show()

In [ ]:
corr = df.select_dtypes(include='number').corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap - Nigeria')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x='T2M', y='RH2M', ax=axes[0])
axes[0].set_title('T2M vs RH2M')
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', ax=axes[1])
axes[1].set_title('T2M_RANGE vs WS2M')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df['PRECTOTCORR'], bins=50)
plt.yscale('log')
plt.title('PRECTOTCORR Distribution (log-y) - Nigeria')
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='T2M', y='RH2M', size='PRECTOTCORR', sizes=(10, 200), alpha=0.5)
plt.title('Bubble: T2M vs RH2M (size = PRECTOTCORR) - Nigeria')
plt.show()